[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/grabuffo/brain-modeling-course/blob/main/notebooks/D2_C5_NB10_perturbing_the_model.ipynb)

# Day 2 · Chapter 5 · Notebook 10 — A virtual lesion: a local change, a whole-brain effect

**Goal.** A model lets us change one region and watch the whole brain respond. We take the mouse model of NB09 at its working
point ($G=0.6$) and change the excitability $\eta$ of one region, the **anterior cingulate cortex** (a cortical hub, 4 nodes):
* **turned down** ($\eta=-6$): as a lesion or a chemogenetic silencing would do;
* **turned up** ($\eta=-3.6$).

Then we compare activity, functional connectivity (FC) and its dynamics (dFC) with the intact brain ($\eta=-4.6$ everywhere).
This is one example of the kind of experiment done in real and virtual mice by Rabuffo et al. (2025, *PNAS*).

The three 10-minute simulations are precomputed (`dev/cache_nb10.py`). They start from the same state and receive the same
noise, so the only difference between them is the change of excitability.

In [ ]:
# --- Setup (run once) -------------------------------------------------------
# On Google Colab this downloads the course code; on your laptop it does nothing.
import sys, os
if "google.colab" in sys.modules and not os.path.exists("brain-modeling-course"):
    !git clone -q https://github.com/grabuffo/brain-modeling-course.git
    %cd brain-modeling-course/notebooks
    !pip -q install numba ipywidgets
sys.path.insert(0, os.path.abspath(".."))   # so that `import src` works

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact
from src import plotting
from src.plotting import slider
plotting.setup()
import pandas as pd
from src import connectome
sc, labels, centres, cortex = connectome.load_mouse()
les = dict(np.load("../data/cache/nb10_lesions.npz"))
CONDS = ["intact", "cingulate_down", "cingulate_up"]
NAMES = {"intact": "intact", "cingulate_down": "cingulate ↓", "cingulate_up": "cingulate ↑"}
COLORS = {"intact": "0.5", "cingulate_down": "C0", "cingulate_up": "C3"}
target = les["cingulate"]

# order the 50 cortical regions by resting-state network, as in NB09
nets = pd.read_csv("../data/mouse/Allen_rois_nets.csv")["Network"].to_numpy()[cortex]
hemi = np.array(["R" if l.startswith("Right") else "L" for l in labels[cortex]])
NETS = ["Visual-auditory", "Laterocortical", "Default mode Network", "Basofrontal"]
order = np.concatenate([np.where((nets == n) & (hemi == h))[0] for n in NETS for h in "RL"])
bounds = np.r_[0, np.cumsum([(nets == n).sum() for n in NETS])]
ticks, tick_names = (bounds[:-1] + bounds[1:]) / 2 - 0.5, ["Vis", "Lat", "DMN", "BF"]

def upper(M):
    return M[np.triu_indices_from(M, 1)]

def edge_dfc(y):
    # edge time series z_i z_j, then correlation between the edge patterns at every pair of times
    z = (y - y.mean(0)) / y.std(0); i, j = np.triu_indices(y.shape[1], 1)
    return np.corrcoef(z[:, i] * z[:, j])

def show_fc(a_, M, title, cmap="RdBu_r", lim=0.8):
    im = a_.imshow(M[np.ix_(order, order)], cmap=cmap, vmin=-lim, vmax=lim); a_.set_title(title)
    for x in bounds[1:-1]: a_.axhline(x - 0.5, c="k", lw=0.6); a_.axvline(x - 0.5, c="k", lw=0.6)
    a_.set_xticks(ticks, tick_names, fontsize=8); a_.set_yticks(ticks, tick_names, fontsize=8)
    return im

FC = {c: np.corrcoef(les[f"bold_{c}"][:, cortex].T) for c in CONDS}
DFC = {c: edge_dfc(les[f"bold_{c}"][:, cortex]) for c in CONDS}

## 1. A local change alters activity far away

How much time does each region spend in the up state, compared with the intact brain? Each dot is a brain region seen from
above; crosses mark the 4 cingulate nodes whose excitability was changed.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
other = np.setdiff1d(np.arange(148), target)
for a_, c in zip(ax, CONDS[1:]):
    d = les[f"up_{c}"] - les["up_intact"]
    sca = a_.scatter(centres[other, 0], centres[other, 1], c=d[other], cmap="RdBu_r", vmin=-0.5, vmax=0.5, s=55, edgecolor="k", lw=0.3)
    a_.scatter(centres[target, 0], centres[target, 1], marker="x", c="k", s=60, label="changed regions")
    a_.set(title=f"{NAMES[c]}: change in time spent up", xticks=[], yticks=[], aspect="equal"); a_.legend(fontsize=8, loc="lower left")
plt.colorbar(sca, ax=ax, shrink=0.8, label="change in fraction of time up")
for c in CONDS[1:]:
    n = np.sum(np.abs(les[f"up_{c}"] - les["up_intact"])[other] > 0.05)
    print(f"{NAMES[c]}: {n} of the {len(other)} other regions changed their time up by more than 5%")

Changing 4 nodes changes the activity of many **other** regions, some of them far away: the change propagates through the
connectome.

## 2. Functional connectivity: down → more FC, up → less FC

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 4.3), gridspec_kw=dict(width_ratios=[1, 1, 1, 0.7]))
for a_, c in zip(ax, CONDS):
    im = show_fc(a_, FC[c], f"FC: {NAMES[c]}")
plt.colorbar(im, ax=ax[2], shrink=0.8)
mean_fc = [upper(FC[c]).mean() for c in CONDS]
ax[3].bar(range(3), mean_fc, color=[COLORS[c] for c in CONDS])
ax[3].set_xticks(range(3), [NAMES[c] for c in CONDS]); ax[3].set(ylabel="mean FC", title="average correlation")
plt.tight_layout()

Turning the cingulate **down** makes the cortex **more** correlated; turning it **up** makes it slightly **less** correlated.
A simple way to see it: with the hub down, the other regions are up less often; when an up-switch does happen it is a large, shared event, and
shared events make correlations. With the hub up, regions switch more often and more independently.

## 3. The dynamics of FC change too

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(18, 4), gridspec_kw=dict(width_ratios=[1, 1, 1, 0.8]))
for a_, c in zip(ax, CONDS):
    a_.imshow(DFC[c][:300, :300], cmap="Purples", vmin=0, vmax=0.5); a_.set(title=f"dFC: {NAMES[c]}", xlabel="time (s)")
ax[0].set_ylabel("time (s)")
var = [upper(DFC[c]).std() for c in CONDS]
ax[3].bar(range(3), var, color=[COLORS[c] for c in CONDS])
ax[3].set_xticks(range(3), [NAMES[c] for c in CONDS]); ax[3].set(ylabel="dFC variability (std)", title="how much FC fluctuates")
plt.tight_layout()

Not only the average correlations change: with the cingulate down, the time × time dFC matrix has more strong blocks
(recurring co-activation patterns) and dFC is more variable; with the cingulate up it is close to the intact brain.

### Explore: compare with the intact brain

In [ ]:
@interact(condition=["cingulate ↓", "cingulate ↑"])
def explore_lesion(condition):
    c = [k for k, v in NAMES.items() if v == condition][0]
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
    show_fc(ax[0], FC["intact"], "FC intact")
    show_fc(ax[1], FC[c], f"FC {condition}")
    im = show_fc(ax[2], FC[c] - FC["intact"], "change", cmap="PuOr_r", lim=0.5)
    plt.colorbar(im, ax=ax[2], shrink=0.8); plt.tight_layout(); plt.show()

## 4. Take-home

* Changing the excitability of one region changes the activity of many distant regions.
* Turning a hub down increases FC; turning it up decreases it. The dynamics of FC (dFC) change as well.
* A model lets us run such experiments for any region and any amount of change; in animals each case needs its own
  intervention.
* **Next (Chapter 6):** a different kind of brain model, learned directly from data.

### Homework
*To do at home, after the class. In class we use the **Explore** menu; these exercises take the same questions further and
usually need a few lines of your own code.*
1. **Another region.** Choose another region or group (e.g. network "Thalamus" or "Hippocampus" in `Allen_rois_nets.csv`), change
   its $\eta$ down or up, and simulate 2 minutes at $G=0.6$:
   ```python
   from src import wholebrain, bold
   eta = np.full(148, -4.6); eta[my_regions] = -5.5
   rng = np.random.default_rng(1)
   t, r, v = wholebrain.mpr_network(sc, 0.6, T=140_000.0, rec_dt=5.0, eta=eta, noise_r=0.01, noise_v=0.02, seed=1,
                                    r0=rng.uniform(0, 2, 148), v0=rng.uniform(-2, 1.5, 148))
   y = bold.downsample(bold.balloon_windkessel(r[t > 20_000], 5e-3), 5e-3, 1.0)[20:]
   ```
   Does FC go up or down? Is the effect the same as for the cingulate?
2. **Dose–response.** For the cingulate, try $\eta = -5.2, -5.6, -6.0$. Does the mean FC change smoothly with the dose?

### References
* Rabuffo G, Lokossou H-A, Li Z, et al. (2025). Mapping global brain reconfigurations following local targeted manipulations. *PNAS* 122:e2405706122.
* Rabuffo G, Fousek J, Bernard C, Jirsa V (2021). Neuronal cascades shape whole-brain functional dynamics at resting state. *eNeuro* 8:ENEURO.0283-21.2021.